In [ ]:
# Locate the project when Jupyter starts in the root or notebooks directory.
from pathlib import Path
import os
import sys

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "project_paths.py").is_file()
)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_paths import DATA_DIR, MODELS_DIR, OUTPUT_DIR


# 5.3.1 改进文本生成策略

按照 **贪心解码 → 随机采样 → Temperature → Top-k → 完整生成函数 → GPT-2 测试** 整理。

## 1. 初始化示例词表与 Logits

这个 9 个 Token 的词表只用于演示采样策略，与后面的 GPT-2 词表（50,257 个 Token）不同。

In [ ]:
import torch
import matplotlib.pyplot as plt

vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)
probas = torch.softmax(next_token_logits, dim=0)

## 2. 贪心解码与随机采样

`argmax` 选择最大概率的 Token；`multinomial` 按概率随机抽取。

In [ ]:
# 贪心解码
next_token_id = torch.argmax(probas).item()
print("Greedy:", inverse_vocab[next_token_id])

# 随机采样一次
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print("Sampled:", inverse_vocab[next_token_id])


def print_sampled_tokens(probas):
    """从一维概率分布中独立采样 1000 次，统计各 Token 出现次数。"""
    torch.manual_seed(123)
    samples = [
        torch.multinomial(probas, num_samples=1).item()
        for _ in range(1_000)
    ]
    # minlength 确保未被采到的词（包括最后一个词）也显示 0 次
    sampled_ids = torch.bincount(torch.tensor(samples), minlength=len(vocab))
    for token_id, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[token_id]}")


print_sampled_tokens(probas)

## 3. Temperature 缩放与概率分布可视化

`T=1` 保持原分布，`T<1` 更集中，`T>1` 更平坦。

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


temperatures = [1, 0.1, 5]
scaled_probas = [
    softmax_with_temperature(next_token_logits, T)
    for T in temperatures
]

x = torch.arange(len(vocab))
bar_width = 0.15
fig, ax = plt.subplots(figsize=(5, 3))

for i, T in enumerate(temperatures):
    ax.bar(
        x + i * bar_width,
        scaled_probas[i],
        width=bar_width,
        label=f"Temperature = {T}"
    )

ax.set_ylabel("Probability")
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

### 练习 5.1：对比 pizza 的采样频率与理论概率

In [ ]:
for T, temperature_probas in zip(temperatures, scaled_probas):
    print(f"Temperature = {T}")
    print_sampled_tokens(temperature_probas)

for T, temperature_probas in zip(temperatures, scaled_probas):
    pizza_prob = temperature_probas[vocab["pizza"]].item()
    print(f"T = {T}: pizza probability = {pizza_prob:.8%}")

## 4. Top-k：仅保留最大的 k 个 Logits

未被保留的位置设置为 `-inf`，经过 Softmax 后概率为 0。

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
print("Top logits:", top_logits)
print("Top positions:", top_pos)

new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")),
    other=next_token_logits
)
print("New logits:", new_logits)

topk_probas = torch.softmax(new_logits, dim=0)
print("Top probas:", topk_probas)

## 5. 改进后的 `generate()` 函数

`idx` 的形状是 `[B, T]`：**B 是当前批次中的样本数，T 是每个样本的 Token 数**。原书例子使用 `B=1`。

这里将 `min_val` 保留为 `[B,1]`，使每个样本的 Top-k 阈值可以正确广播到 `[B, vocab_size]`。

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        # [B, T] -> [B, min(T, context_size)]
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        # [B, T, vocab_size] -> [B, vocab_size]
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k, dim=-1)
            min_val = top_logits[:, -1].unsqueeze(-1)  # [B, 1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf"), device=logits.device),
                logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # [B, 1]
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # [B, 1]

        # 原书中的单样本生成终止逻辑；多样本需分别处理结束状态
        if eos_id is not None and idx.size(0) == 1 and idx_next.item() == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)  # [B, T+1]

    return idx

## 6. 在 GPT-2 124M 上测试

下方使用本地 `chapter04.py` 中的 `GPTModel`。**这里没有加载预训练权重**，因此模型仍是随机初始化，文本通常没有语义连贯性。

In [ ]:
import tiktoken
from chapter04 import GPTModel

tokenizer = tiktoken.get_encoding("gpt2")


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)  # [1, T]


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

torch.manual_seed(123)
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))